# 🎗️ 01 · Conociendo CBIS-DDSM

**Lectura breve · 10–15 minutos**

Este cuaderno muestra cómo está organizado el conjunto de datos que usaremos en el taller. Léelo en orden y ejecuta las celdas para ver los recuentos. **No tienes que responder preguntas ni modificar el código.**

Aquí no se muestran las mamografías ni los resultados individuales de los tres casos del siguiente cuaderno. [Fuente oficial: The Cancer Imaging Archive](https://www.cancerimagingarchive.net/collection/cbis-ddsm/).


## 1. Prepara los archivos

En este proyecto, los CSV están en `data/`. En Google Colab, la celda pedirá subir `taller_colab.zip`. Ese paquete trae los CSV y los tres pares DICOM elegidos para el taller; **no contiene todo CBIS-DDSM**.


In [ ]:
from pathlib import Path
import csv
import io
import zipfile

import matplotlib.pyplot as plt

ubicaciones = [Path("data"), Path("../data"), Path("/content/data")]
carpeta_datos = next(
    (ruta for ruta in ubicaciones if (ruta / "metadata.csv").exists()),
    None,
)

if carpeta_datos is None:
    try:
        from google.colab import files
    except ImportError as error:
        raise FileNotFoundError("No encuentro la carpeta data del taller.") from error

    print("Sube el archivo taller_colab.zip")
    archivos = files.upload()
    nombre_zip = next((nombre for nombre in archivos if nombre.endswith(".zip")), None)
    if nombre_zip is None:
        raise ValueError("Se necesita taller_colab.zip para continuar.")

    with zipfile.ZipFile(io.BytesIO(archivos[nombre_zip])) as paquete:
        rutas = [Path(nombre) for nombre in paquete.namelist()]
        if any(ruta.is_absolute() or ".." in ruta.parts for ruta in rutas):
            raise ValueError("El ZIP contiene rutas no válidas.")
        paquete.extractall("/content")

    carpeta_datos = Path("/content/data")

print("Archivos listos en:", carpeta_datos)


## 2. Las tres piezas que usaremos

| Archivo | Qué contiene | Para qué lo usamos |
|---|---|---|
| **Mamografía DICOM** | La imagen completa | Observar el caso |
| **Máscara DICOM** | Una región marcada como referencia | Comparar nuestra observación |
| **CSV de descripción** | Datos del hallazgo y rutas de archivos | Entender de qué caso se trata |

La **máscara es una anotación de referencia**. No es una detección hecha por un modelo de IA. Una mamografía puede aparecer en varias filas si tiene más de un hallazgo anotado.


## 3. Los cuatro CSV de casos

Los archivos separan **masas** y **calcificaciones**. Cada grupo tiene una parte de **entrenamiento** y otra de **prueba**. En la tabla que imprimirá la celda, «filas» significa registros de hallazgos, no personas.


In [ ]:
def leer_csv(ruta):
    with ruta.open(encoding="utf-8-sig", newline="") as archivo:
        return list(csv.DictReader(archivo))

archivos_casos = sorted(carpeta_datos.glob("*_case_description_*_set.csv"))
filas_por_archivo = {ruta.name: leer_csv(ruta) for ruta in archivos_casos}

for nombre, filas in filas_por_archivo.items():
    print(f"{nombre:38} {len(filas):4} filas")


Una fila de estos CSV registra **un hallazgo anotado en una vista**. Por eso el número de filas no sirve para decir cuántas personas hay. El `patient_id` ayuda a relacionar registros, y una misma persona puede tener varias vistas o anomalías.

Las columnas incluyen lado, vista, tipo de hallazgo y, según el archivo, forma o distribución. También existe una etiqueta de patología, que **no mostraremos por caso** antes del ejercicio práctico. Los nombres de algunas columnas varían entre los CSV de masas y calcificaciones.


## 4. El catálogo de imágenes

`metadata.csv` cumple otra función: cataloga **series DICOM**. El gráfico siguiente muestra cuántas series están descritas como mamografía completa, máscara o recorte. Cuenta series del catálogo, **no pacientes ni diagnósticos**.


In [ ]:
from collections import Counter

filas_metadata = leer_csv(carpeta_datos / "metadata.csv")

nombres = {
    "full mammogram images": "Mamografías completas",
    "ROI mask images": "Máscaras",
    "cropped images": "Recortes",
}

conteo = Counter(fila["Series Description"] for fila in filas_metadata)
etiquetas = list(nombres.values())
cantidades = [conteo[nombre] for nombre in nombres]

figura, eje = plt.subplots(figsize=(7, 3.5))
eje.barh(etiquetas, cantidades, color="#5587a2")
eje.set_xlabel("Series registradas")
eje.set_title("Tipos de serie en metadata.csv")
eje.invert_yaxis()
plt.tight_layout()
plt.show()


Las barras no tienen por qué ser iguales: **una mamografía y una máscara no son la misma clase de archivo**, y puede haber varios registros asociados a un caso. El catálogo tampoco se interpreta como una lista de personas.


## 5. Entrenamiento y prueba

Los CSV distinguen una parte para desarrollar métodos (*train*) y otra para comprobarlos (*test*). El gráfico compara el número de **filas de descripción** de cada tipo. Una barra más alta **no indica mayor frecuencia de enfermedad en la población**; solo describe estos archivos.


In [ ]:
tipos = ["masa", "calcificaciones"]
entrenamiento = [
    len(filas_por_archivo[f"{nombre}_case_description_train_set.csv"])
    for nombre in ("mass", "calc")
]
prueba = [
    len(filas_por_archivo[f"{nombre}_case_description_test_set.csv"])
    for nombre in ("mass", "calc")
]

posiciones = [0, 1]
ancho = 0.35

figura, eje = plt.subplots(figsize=(7, 4))
eje.bar([x - ancho / 2 for x in posiciones], entrenamiento,
        width=ancho, label="Entrenamiento", color="#4d7893")
eje.bar([x + ancho / 2 for x in posiciones], prueba,
        width=ancho, label="Prueba", color="#d5a05a")
eje.set_xticks(posiciones, tipos)
eje.set_ylabel("Filas de descripción")
eje.set_title("Particiones de los CSV de casos")
eje.legend()
plt.tight_layout()
plt.show()


En un proyecto de entrenamiento, la parte de prueba se reserva para evaluar el resultado final. Además, hay que revisar que las imágenes de una misma persona no terminen mezcladas entre entrenamiento y evaluación.


## 6. Un DICOM por dentro

El DICOM guarda los píxeles y una **cabecera** con datos técnicos. Leeremos la cabecera de una copia del taller, sin mostrar la imagen, la máscara ni el resultado de ese caso.


In [ ]:
import subprocess
import sys

try:
    import pydicom
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "pydicom>=3,<4", "-q"
    ])
    import pydicom

ruta_dicom = carpeta_datos / "taller" / "dicom" / "masa_1_imagen.dcm"
dicom = pydicom.dcmread(ruta_dicom, stop_before_pixels=True)

print("Modalidad:", dicom.get("Modality", "sin dato"))
print("Tamaño:", dicom.Rows, "filas ×", dicom.Columns, "columnas")
print("Bits guardados por píxel:", dicom.BitsStored)


La cabecera describe el archivo; no dice dónde mirar en la imagen. En el próximo cuaderno veremos la mamografía completa y, después de observarla, revelaremos su máscara.

**En resumen:** CBIS-DDSM reúne imágenes DICOM, anotaciones y descripciones de casos. Una **fila**, una **serie** y una **persona** son unidades distintas. Los recuentos de este cuaderno describen los CSV incluidos en el proyecto; no estiman cuántas personas tienen cáncer.

**Fuente y licencia:** [CBIS-DDSM, The Cancer Imaging Archive](https://www.cancerimagingarchive.net/collection/cbis-ddsm/) (CC BY 3.0).
